In [3]:
import pandas as pd
import os

# --- Path Setup ---
# The location of your dataset relative to your current script or via absolute path
# Adjust 'base_path' if the script is being run from a different drive/root
base_path = r'C:\Users\Jalynn\OneDrive\Documents\GitHub\Universal_EEG_Analyzer\StudyTwoGermaneLoadAnalysis\GammaResults2'

# Where you want to save the 4 new files
output_dir = os.path.join(os.getcwd(), 'Extracted_Features')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# The four specific files we are looking for in each participant folder
target_files = [
    "LowGermane_psd_results_train.csv",
    "LowGermane_psd_results_test.csv",
    "HighGermane_psd_results_train.csv",
    "HighGermane_psd_results_test.csv"
]

# Initialize containers
data_accumulators = {name: [] for name in target_files}

print(f"Searching in: {base_path}")

# --- Processing ---
if not os.path.exists(base_path):
    print(f"Error: Could not find the directory {base_path}. Please check the path.")
else:
    # Loop through every participant folder (e.g., P01, P02...)
    for participant_id in os.listdir(base_path):
        participant_path = os.path.join(base_path, participant_id)
        
        if os.path.isdir(participant_path):
            for file_name in target_files:
                file_path = os.path.join(participant_path, file_name)
                
                if os.path.exists(file_path):
                    try:
                        df = pd.read_csv(file_path)
                        
                        # Extract Pz Alpha and Fz Theta
                        # Using .get() or .iloc[0] safely
                        pz_row = df[df['Channel'] == 'Pz']
                        fz_row = df[df['Channel'] == 'Fz']
                        
                        if not pz_row.empty and not fz_row.empty:
                            pz_alpha = pz_row['psd_Alpha'].values[0]
                            fz_theta = fz_row['psd_Theta'].values[0]
                            
                            data_accumulators[file_name].append({
                                'PID': participant_id,
                                'Pz_Alpha': pz_alpha,
                                'Fz_Theta': fz_theta
                            })
                    except Exception as e:
                        print(f"Error in {participant_id} for {file_name}: {e}")

# --- Exporting Results ---
for file_name, data_list in data_accumulators.items():
    if data_list:
        final_df = pd.DataFrame(data_list)
        # Create a clean output name (e.g., Summary_LowGermane_psd_results_train.csv)
        output_name = f"Summary_{file_name}"
        save_path = os.path.join(output_dir, output_name)
        
        final_df.to_csv(save_path, index=False)
        print(f"Successfully created: {output_name} with {len(final_df)} entries.")
    else:
        print(f"Warning: No data collected for {file_name}")

print("\nProcessing Complete.")

Searching in: C:\Users\Jalynn\OneDrive\Documents\GitHub\Universal_EEG_Analyzer\StudyTwoGermaneLoadAnalysis\GammaResults2
Successfully created: Summary_LowGermane_psd_results_train.csv with 34 entries.
Successfully created: Summary_LowGermane_psd_results_test.csv with 34 entries.
Successfully created: Summary_HighGermane_psd_results_train.csv with 34 entries.
Successfully created: Summary_HighGermane_psd_results_test.csv with 34 entries.

Processing Complete.
